# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Shape and memory:** After loading and schema alignment, both datasets have 77 columns (76 features + Label). The training set (CIC-IDS2017, 10% sample) holds 281,434 rows at 159.7 MB; the progressive test set (CSE-CIC-IDS2018) holds 124,696 rows at 70.9 MB — comfortably within free Colab RAM.

**Dtype breakdown:** Both datasets share the same pattern: 52 `int64` columns, 24 `float32`, and 1 `object` (Label). The split makes sense — flag counts and raw packet/byte counts are inherently discrete integers and were never cast. Rates, averages, standard deviations and variances are continuous and were downcast from `float64` to `float32`, halving their memory footprint with negligible precision loss for ML.

**Feature groups (76 features across 5 semantic groups):**
- **Packet length stats (16):** Min, max, mean, std, variance of packet sizes in both forward and backward direction, plus overall flow statistics. Capture *what is being sent* — attack flows often show extremely uniform sizes (e.g. flooding with fixed-size packets) or extreme values.
- **Packet counts and byte rates (18):** Total packet/byte counts, flow-level rates (packets/s, bytes/s), subflow counts, bulk transfer stats. Capture *volume and asymmetry* — DoS floods show extreme forward rates with near-zero backward traffic; exfiltration shows the reverse.
- **Inter-arrival times (15):** Min, max, mean, std of gaps between packets within a flow, plus total flow duration. Capture *timing patterns* — port scans and flooding attacks have unusually regular or extremely small IATs compared to human-generated traffic.
- **TCP flags (14):** Counts of SYN/FIN/RST/PSH/ACK/URG/ECE/CWE flags, plus initial TCP window sizes (forward and backward). Capture *connection behaviour* — a high SYN count with no matching ACK is a textbook SYN flood; window sizes can fingerprint operating systems and reveal botnet clients.
- **Other / port / misc (13):** Destination port, header lengths, active/idle time statistics, and two CICFlowMeter-specific fields (`act_data_pkt_fwd`, `min_seg_size_forward`). Destination port alone is a strong discriminator — web traffic concentrates on 80/443, SSH on 22, scanning generates traffic across unusual high ports.

**Labels:** The training set contains 15 distinct attack types plus BENIGN. The test set contains 10 attack types plus BENIGN, with all labels now decoded to human-readable strings (2018 used integer encoding in the pre-processed file; 2017 had UTF-8 encoding artifacts in the Web Attack labels — both fixed at load time).

**No timestamp column in features:** CICFlowMeter writes a `Timestamp` field to the raw CSVs, but it is deliberately excluded from the feature matrix. Using it would constitute time leakage — a model could learn "flows from 2018 = test distribution" rather than genuine traffic patterns. The temporal structure is enforced at the dataset-split level, not the feature level.

### Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type coverage in each dataset ─────────────────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
train_labels = df_train['Label'].value_counts()
print(train_labels.to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
test_labels = df_test['Label'].value_counts()
print(test_labels.to_string())

# ── Semantic grouping — map both datasets to a shared attack family ───────────
# The two datasets use different naming conventions for the same attack families.
# We normalise to a common family name before comparing.
_FAMILY = {
    # 2017 names
    'BENIGN':                    'Benign',
    'Bot':                       'Bot',
    'DDoS':                      'DDoS',
    'PortScan':                  'PortScan',
    'FTP-Patator':               'Brute Force - FTP',
    'SSH-Patator':               'Brute Force - SSH',
    'DoS slowloris':             'DoS - Slowloris',
    'DoS Slowhttptest':          'DoS - SlowHTTPTest',
    'DoS Hulk':                  'DoS - Hulk',
    'DoS GoldenEye':             'DoS - GoldenEye',
    'Heartbleed':                'Heartbleed',
    'Infiltration':              'Infiltration',
    'Web Attack - Brute Force':  'Brute Force - Web',
    'Web Attack - XSS':          'Brute Force - XSS',
    'Web Attack - Sql Injection':'SQL Injection',
    # 2018 names
    'Brute Force - Web':         'Brute Force - Web',
    'Brute Force - XSS':         'Brute Force - XSS',
    'DDoS - HOIC':               'DDoS',
    'DDoS - LOIC-UDP':           'DDoS',
    'DDoS - LOIC-HTTP':          'DDoS',
    'DoS - GoldenEye':           'DoS - GoldenEye',
    'DoS - Hulk':                'DoS - Hulk',
    'DoS - SlowHTTPTest':        'DoS - SlowHTTPTest',
    'DoS - Slowloris':           'DoS - Slowloris',
}

train_families = set(df_train['Label'].map(_FAMILY).dropna().unique())
test_families  = set(df_test['Label'].map(_FAMILY).dropna().unique())

in_both     = train_families & test_families
only_train  = train_families - test_families
only_test   = test_families  - train_families

print(f"\nAttack families in BOTH datasets:        {sorted(in_both)}")
print(f"Attack families ONLY in train (2017):    {sorted(only_train)}")
print(f"Attack families ONLY in test  (2018):    {sorted(only_test)}")

**Temporal structure — what time means in this project:**

**1. Within each dataset (daily granularity):**
CIC-IDS2017 is split across 8 CSV files covering five working days (Monday 3 July – Friday 7 July 2017). Monday is benign-only background traffic. From Tuesday onward, increasingly complex attacks are injected: Tuesday has brute-force (FTP-Patator, SSH-Patator); Wednesday has DoS/DDoS; Thursday has Web Attacks and Infiltration; Friday has DDoS and PortScan. CSE-CIC-IDS2018 arrives as a single pre-processed file — the within-day structure has been collapsed.

**2. Between datasets — the progressive gap (~8 months):**
All 2017 data precedes all 2018 data. There is zero temporal overlap. This is the core of the paper's methodology: a model trained on July 2017 traffic is evaluated on February–March 2018 traffic it has never seen.

**Class imbalance in both datasets:**
- Train: 226,117 BENIGN vs 55,317 attacks → **80.4% benign**
- Test: 96,421 BENIGN vs 28,475 attacks → **77.2% benign**

Both datasets reflect realistic network conditions where the vast majority of traffic is benign. This imbalance is a central methodological issue — the authors address it by downsampling to a 1:1 ratio before training.

**Attack type coverage and concept drift:**
Only two classes appear in both datasets by name: **BENIGN** and **Bot**. Everything else has diverged — but not entirely in meaning, only partly in naming:

- *Semantically overlapping but differently named:* "DoS Hulk" (2017) ≈ "DoS - Hulk" (2018); "DDoS" (2017) ≈ "DDoS - HOIC / LOIC-HTTP / LOIC-UDP" (2018). The attack families are the same but the 2018 file uses more specific tool names. A model trained on 2017 DDoS flows may still detect 2018 DDoS flows if the underlying traffic pattern is similar — or it may fail if the specific tool generates a different flow signature.

- *Genuinely absent in 2018:* PortScan, Heartbleed, FTP-Patator, SSH-Patator, Infiltration, and the generic Web Attack variants. The model has no positive examples of these attack types at test time — but since we use binary labels this only matters for multiclass analysis.

- *Genuinely new in 2018:* DDoS - LOIC-HTTP, DDoS - LOIC-UDP, DDoS - HOIC, Brute Force - Web, Brute Force - XSS. These specific tools were not present in training. The model must generalise from the 2017 attack patterns to detect them — this is exactly the concept drift the paper studies.

**This distribution shift — not classical overfitting — is the primary driver of the performance drop reported in the paper.** A model that memorises 2017 attack signatures without learning transferable flow-level patterns will degrade on 2018 traffic. We will revisit this in Phase 8 (overfitting vs. concept drift analysis).

**Leakage check:** No temporal leakage exists in our setup. Timestamp is excluded from features; the train/test split is fixed at the dataset level; no 2018 samples appear in training.

### Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

**Hygiene findings:**

**Duplicates:** 14,695 rows removed from train (5.2%) and 17,790 from test (14.3%). Duplicate flows in network data are common — multiple identical flows can be generated when two different attack scripts hit the same destination with identical parameters, or when benign background applications repeatedly open connections with the same characteristics. The higher rate in the test set likely reflects the more uniform traffic patterns in the pre-processed 2018 file.

**Constant columns (10 dropped):** The following columns were zero across every row in at least one dataset and were removed: `Bwd Avg Bulk Rate`, `Bwd Avg Bytes/Bulk`, `Bwd Avg Packets/Bulk`, `Fwd Avg Bulk Rate`, `Fwd Avg Bytes/Bulk`, `Fwd Avg Packets/Bulk`, `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd URG Flags`, `CWE Flag Count`. This is a known limitation of CICFlowMeter on the CIC datasets — bulk transfer statistics are only populated when a flow is identified as a bulk transfer (a specific heuristic), which rarely triggers in simulated lab traffic. These features carry zero information and would only add noise to a model.

**Near-constant but retained:** `ECE Flag Count` and `RST Flag Count` exceed the 99.9% threshold in training but are not constant in the test set. ECE (Explicit Congestion Notification) is rarely set in lab traffic but can appear in real or attack flows; RST (connection reset) is occasionally triggered by scanner tools. They are kept as features since their rare non-zero values may still be informative.

**After cleaning:** Train: 266,739 rows × 66 features. Test: 106,906 rows × 66 features. Both saved to Drive as `train_clean.joblib` / `test_clean.joblib` for use in all subsequent phases.

In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Class distribution, feature distributions, missing values, outliers, temporal patterns, correlation analysis.

---
## §3 — Feature Engineering

Reproduce the authors' pipeline: cleaning → balancing → binary relabeling → encoding → scaling → feature creation → feature selection.

---
## §4 — Model Training

Train DT, RF, SVM, NB, ANN, DNN with GridSearchCV (k=5). Save best models to Drive.

---
## §5 — Evaluation & Reproduction Check

In-distribution evaluation (reproduce Tables 4–6). Progressive evaluation on CSE-CIC-IDS2018 (reproduce Table 7). Side-by-side comparison with paper's numbers.

---
## §6 — Error Analysis

Misclassified examples (FPs and FNs) on the progressive test set. Patterns in errors. Cybersecurity implications.

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*